# 04 — 模型训练
**4个模型对比：Random Forest / XGBoost / 2D CNN / U-Net**

特征：2024年4月 Sentinel-2 月合成（NDVI, NDWI, BSI, SAVI, B04, B08, B11）
标签：2024年4月 NSW SEED 裸土侵蚀量（t/ha/月）

时间完全匹配，用于评估各模型真实精度。

## 1. 导入包

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import time
import joblib
import subprocess
warnings.filterwarnings('ignore')

from src.data_utils import load_config, ensure_dir
from src.metrics import evaluate_all

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')

PyTorch : 2.5.1+cu121
CUDA    : True
GPU     : NVIDIA GeForce RTX 3090


## 2. 加载特征与标签（2024年4月，时间匹配）

In [2]:
cfg           = load_config('../configs/config.yaml')
processed_dir = Path('../data/processed')
model_dir     = ensure_dir('../models')
fig_dir       = ensure_dir('../results/figures')
LABEL         = '2024_04'

# 加载特征矩阵和标签（时间完全匹配）
features = np.load(processed_dir / f'NSW_test_features_{LABEL}.npy')         # (H, W, 7)
labels   = np.load(processed_dir / f'NSW_test_label_ero_{LABEL.replace("_","")}.npy')  # (H, W)

print(f'特征矩阵形状: {features.shape}')
print(f'标签形状    : {labels.shape}')
print(f'特征通道    : NDVI, NDWI, BSI, SAVI, B04, B08, B11')
print(f'时间匹配    : ✓ 2024年4月')

# 只保留特征和标签都有效的像素
valid_mask = np.isfinite(labels) & np.all(np.isfinite(features), axis=-1)
print(f'\n有效像素数  : {valid_mask.sum():,}')
print(f'有效像素比例: {valid_mask.mean()*100:.1f}%')

X = features[valid_mask]  # (N, 7)
y = labels[valid_mask]    # (N,)

print(f'\nX 形状: {X.shape}')
print(f'y 形状: {y.shape}')
print(f'y 范围: {y.min():.4f} ~ {y.max():.4f} t/ha/月')
print(f'y 均值: {y.mean():.4f}')

特征矩阵形状: (16811, 18790, 7)
标签形状    : (16811, 18790)
特征通道    : NDVI, NDWI, BSI, SAVI, B04, B08, B11
时间匹配    : ✓ 2024年4月

有效像素数  : 309,904,786
有效像素比例: 98.1%

X 形状: (309904786, 7)
y 形状: (309904786,)
y 范围: 0.0414 ~ 637.3094 t/ha/月
y 均值: 7.4004


## 3. 采样与划分训练/验证/测试集

In [3]:
from sklearn.model_selection import train_test_split

N_SAMPLES = 1_000_000
np.random.seed(42)
idx      = np.random.choice(len(X), size=N_SAMPLES, replace=False)
X_sample = X[idx]
y_sample = y[idx]

# 划分 70 / 15 / 15
X_train, X_temp, y_train, y_temp = train_test_split(
    X_sample, y_sample, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42)

print(f'训练集: {X_train.shape[0]:,} 样本')
print(f'验证集: {X_val.shape[0]:,} 样本')
print(f'测试集: {X_test.shape[0]:,} 样本')

# 对数变换（侵蚀量分布右偏）
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)
y_test_log  = np.log1p(y_test)

print(f'\n对数变换后 y 范围: {y_train_log.min():.3f} ~ {y_train_log.max():.3f}')
print(f'对数变换后 y 均值: {y_train_log.mean():.3f}')

训练集: 700,000 样本
验证集: 150,000 样本
测试集: 150,000 样本

对数变换后 y 范围: 0.041 ~ 6.363
对数变换后 y 均值: 1.225


## 4. 模型 1/4 — Random Forest

In [4]:
from sklearn.ensemble import RandomForestRegressor

print('='*40)
print('模型 1/4 — Random Forest')
print('='*40)

rf = RandomForestRegressor(
    n_estimators=200, max_depth=15,
    min_samples_leaf=4, n_jobs=-1, random_state=42
)

t0 = time.time()
rf.fit(X_train, y_train_log)
print(f'训练时间: {time.time()-t0:.1f} 秒')

y_pred = np.expm1(rf.predict(X_val))
rf_metrics = evaluate_all(y_val, y_pred, label='Random Forest')

feat_names = ['NDVI','NDWI','BSI','SAVI','B04','B08','B11']
print('\n特征重要性:')
for name, imp in sorted(zip(feat_names, rf.feature_importances_), key=lambda x: -x[1]):
    print(f'  {name:<6}: {imp:.4f}')

joblib.dump(rf, model_dir / 'rf_model_2024_04.joblib')
print(f'\n模型已保存: rf_model_2024_04.joblib')

模型 1/4 — Random Forest
训练时间: 28.0 秒

  评估结果 — Random Forest
  RMSE    : +20.1595
  MAE     : +6.3345
  R2      : +0.1118
  Bias    : -4.0942

特征重要性:
  NDVI  : 0.4720
  B11   : 0.2189
  BSI   : 0.0985
  NDWI  : 0.0927
  B04   : 0.0588
  SAVI  : 0.0321
  B08   : 0.0270

模型已保存: rf_model_2024_04.joblib


## 5. 模型 2/4 — XGBoost

In [5]:
import xgboost as xgb

print('='*40)
print('模型 2/4 — XGBoost')
print('='*40)

xgb_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=8,
    learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, tree_method='hist',
    device='cuda', random_state=42
)

t0 = time.time()
xgb_model.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    verbose=100
)
print(f'\n训练时间: {time.time()-t0:.1f} 秒')

y_pred = np.expm1(xgb_model.predict(X_val))
xgb_metrics = evaluate_all(y_val, y_pred, label='XGBoost')

joblib.dump(xgb_model, model_dir / 'xgb_model_2024_04.joblib')
print(f'模型已保存: xgb_model_2024_04.joblib')

模型 2/4 — XGBoost
[0]	validation_0-rmse:1.08564
[100]	validation_0-rmse:0.90353
[200]	validation_0-rmse:0.89979
[300]	validation_0-rmse:0.89795
[400]	validation_0-rmse:0.89705
[499]	validation_0-rmse:0.89626

训练时间: 5.8 秒

  评估结果 — XGBoost
  RMSE    : +20.1525
  MAE     : +6.3218
  R2      : +0.1125
  Bias    : -4.0623
模型已保存: xgb_model_2024_04.joblib


## 6. 采样 Patch（供 2D CNN 和 U-Net 共用）

In [6]:
PATCH_SIZE = 64
PATCH_NUM  = 5000
device     = torch.device('cuda')

H, W, C    = features.shape
patches_X  = []
patches_y  = []

np.random.seed(42)
count = 0
while count < PATCH_NUM:
    r = np.random.randint(0, H - PATCH_SIZE)
    c = np.random.randint(0, W - PATCH_SIZE)
    px = features[r:r+PATCH_SIZE, c:c+PATCH_SIZE, :]
    py = labels[r:r+PATCH_SIZE,   c:c+PATCH_SIZE]
    if np.isfinite(py).mean() > 0.8 and np.isfinite(px).all():
        patches_X.append(px)
        patches_y.append(py)
        count += 1

patches_X = np.stack(patches_X).transpose(0,3,1,2).astype('float32')  # (N,7,64,64)
patches_y = np.log1p(np.nan_to_num(np.stack(patches_y), nan=0)).astype('float32')  # (N,64,64)

split     = int(len(patches_X) * 0.85)
Xp_train, Xp_val = patches_X[:split], patches_X[split:]
yp_train, yp_val = patches_y[:split], patches_y[split:]

print(f'Patch 数量  : {len(patches_X)}')
print(f'训练 patch  : {len(Xp_train)}   验证 patch: {len(Xp_val)}')

Patch 数量  : 5000
训练 patch  : 4250   验证 patch: 750


## 7. 模型 3/4 — 2D CNN

In [ ]:
print('='*40)
print('模型 3/4 — 2D CNN')
print('='*40)

class CNN2D(nn.Module):
    def __init__(self, in_channels=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, padding=1),          nn.ReLU(), nn.BatchNorm2d(64),
            nn.Conv2d(64, 128, 3, padding=1),         nn.ReLU(), nn.BatchNorm2d(128),
            nn.Conv2d(128, 1, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

model_cnn = CNN2D().to(device)
opt_cnn   = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)
sch_cnn   = torch.optim.lr_scheduler.StepLR(opt_cnn, step_size=10, gamma=0.5)
criterion = nn.MSELoss()

dl_tr = DataLoader(TensorDataset(torch.FloatTensor(Xp_train), torch.FloatTensor(yp_train)), batch_size=32, shuffle=True)
dl_vl = DataLoader(TensorDataset(torch.FloatTensor(Xp_val),   torch.FloatTensor(yp_val)),  batch_size=32)

EPOCHS = 30
t0     = time.time()
for epoch in range(EPOCHS):
    model_cnn.train()
    tr_loss = 0
    for xb, yb in dl_tr:
        xb, yb = xb.to(device), yb.to(device)
        opt_cnn.zero_grad()
        loss = criterion(model_cnn(xb), yb)
        loss.backward()
        opt_cnn.step()
        tr_loss += loss.item()
    sch_cnn.step()
    if (epoch+1) % 10 == 0:
        model_cnn.eval()
        vl_loss = sum(criterion(model_cnn(xb.to(device)), yb.to(device)).item() for xb, yb in dl_vl)
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | 训练Loss: {tr_loss/len(dl_tr):.4f} | 验证Loss: {vl_loss/len(dl_vl):.4f}')

print(f'\n训练时间: {time.time()-t0:.1f} 秒')

model_cnn.eval()
y_preds, y_trues = [], []
with torch.no_grad():
    for xb, yb in dl_vl:
        y_preds.append(np.expm1(model_cnn(xb.to(device)).cpu().numpy()).flatten())
        y_trues.append(np.expm1(yb.numpy()).flatten())

cnn_metrics = evaluate_all(np.concatenate(y_trues), np.concatenate(y_preds), label='2D CNN')

torch.save(model_cnn.state_dict(), model_dir / 'cnn2d_model_2024_04.pth')
print(f'模型已保存: cnn2d_model_2024_04.pth')

模型 3/4 — 2D CNN
Epoch  10/30 | 训练Loss: 0.7751 | 验证Loss: 0.7989
Epoch  20/30 | 训练Loss: 0.7325 | 验证Loss: 0.8274


## 8. 模型 4/4 — U-Net

In [ ]:
import segmentation_models_pytorch as smp

print('='*40)
print('模型 4/4 — U-Net')
print('='*40)

unet    = smp.Unet(
    encoder_name='resnet34', encoder_weights=None,
    in_channels=7, classes=1, activation=None
).to(device)

opt_u   = torch.optim.Adam(unet.parameters(), lr=1e-4)
dl_tr_u = DataLoader(TensorDataset(torch.FloatTensor(Xp_train), torch.FloatTensor(yp_train)), batch_size=16, shuffle=True)
dl_vl_u = DataLoader(TensorDataset(torch.FloatTensor(Xp_val),   torch.FloatTensor(yp_val)),  batch_size=16)

EPOCHS_U = 20
t0       = time.time()
for epoch in range(EPOCHS_U):
    unet.train()
    tr_loss = 0
    for xb, yb in dl_tr_u:
        xb, yb = xb.to(device), yb.to(device)
        opt_u.zero_grad()
        loss = criterion(unet(xb).squeeze(1), yb)
        loss.backward()
        opt_u.step()
        tr_loss += loss.item()
    if (epoch+1) % 5 == 0:
        unet.eval()
        vl_loss = sum(criterion(unet(xb.to(device)).squeeze(1), yb.to(device)).item() for xb, yb in dl_vl_u)
        print(f'Epoch {epoch+1:3d}/{EPOCHS_U} | 训练Loss: {tr_loss/len(dl_tr_u):.4f} | 验证Loss: {vl_loss/len(dl_vl_u):.4f}')

print(f'\n训练时间: {time.time()-t0:.1f} 秒')

unet.eval()
y_preds, y_trues = [], []
with torch.no_grad():
    for xb, yb in dl_vl_u:
        y_preds.append(np.expm1(unet(xb.to(device)).squeeze(1).cpu().numpy()).flatten())
        y_trues.append(np.expm1(yb.numpy()).flatten())

unet_metrics = evaluate_all(np.concatenate(y_trues), np.concatenate(y_preds), label='U-Net')

torch.save(unet.state_dict(), model_dir / 'unet_model_2024_04.pth')
print(f'模型已保存: unet_model_2024_04.pth')

## 9. 4个模型精度对比

In [ ]:
results = {
    'Random Forest': rf_metrics,
    'XGBoost':       xgb_metrics,
    '2D CNN':        cnn_metrics,
    'U-Net':         unet_metrics,
}

print(f'{"模型":<15} {"RMSE":>8} {"MAE":>8} {"R²":>8} {"Bias":>8}')
print('-' * 45)
for name, m in results.items():
    print(f'{name:<15} {m["RMSE"]:>8.4f} {m["MAE"]:>8.4f} {m["R2"]:>8.4f} {m["Bias"]:>8.4f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('4个模型精度对比\n（2024年4月特征 vs 2024年4月标签，时间匹配）', fontsize=13)

names   = list(results.keys())
colors  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
metrics = ['RMSE', 'MAE', 'R2']
titles  = ['RMSE（越低越好）', 'MAE（越低越好）', 'R²（越高越好）']

for ax, metric, title in zip(axes, metrics, titles):
    vals = [results[n][metric] for n in names]
    bars = ax.bar(names, vals, color=colors)
    ax.set_title(title, fontsize=11)
    ax.set_xticklabels(names, rotation=15, ha='right')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01 * abs(bar.get_height()),
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig_path = fig_dir / 'model_comparison_2024_04.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'\n图像已保存: {fig_path.name}')
plt.show()

## 10. 提交到 GitHub

In [ ]:
commands = [
    ['git', 'add', 'notebooks/04_model_training.ipynb'],
    ['git', 'add', 'results/figures/model_comparison_2024_04.png'],
    ['git', 'commit', '-m', 'feat: notebook 04 refactor - retrain 4 models with time-matched 2024-04 data'],
    ['git', 'push']
]

for cmd in commands:
    result = subprocess.run(cmd, cwd='..', capture_output=True, text=True)
    print(' '.join(cmd))
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)